# 4. Linear systems and sparse matrices

CS 301 · Unit II essentials practice

**Why we do this.** Linear systems recover unknown quantities from relationships between them. Sparse storage makes this practical when each equation involves only a few unknowns.

**By the end, you should be able to:**

- Build and solve a linear system from a random-walk model.
- Distinguish uniqueness, conditioning, residual, and solution error.
- Read CSR storage, compute a sparse product, and explain fill-in.

Estimated working time: **50–60 minutes**, including written responses.

## Working through this notebook

Save your own copy. Run the setup, then work in order. Before each exercise, write a short prediction. Replace `...` in the exercise cells, run the checks, and explain the result in your own words. A passing check verifies the calculation; the written response shows how you interpret it.

Each notebook runs independently. No results from another notebook are needed. The examples use modest arrays; a GPU is optional in Notebook 3.

**Name:** …


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.set_printoptions(precision=4, suppress=True)

def ready(*values):
    assert all(v is not Ellipsis and v is not None for v in values), "Replace every ... in the exercise before running its check."

def heatmap(A, title, labels=None, limits=None):
    fig, ax = plt.subplots(figsize=(6, 4))
    scale = {} if limits is None else {"vmin": limits[0], "vmax": limits[1]}
    im = ax.imshow(A, cmap="coolwarm", aspect="auto", **scale)
    ax.set_title(title)
    if labels is not None:
        ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
        ax.set_yticks(range(len(labels)), labels)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.show()

from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import spsolve


## 1. Build the equations before solving

Consider the path **red — u — v — blue**. At each step a walker chooses either neighbour with equal probability, stopping when it reaches an endpoint. We want the probabilities of reaching red first. The known endpoint probabilities are 1 at red and 0 at blue.

At each interior node, its probability is the average of its neighbours' probabilities. Write the two equations, multiply each by 2, and move the unknown terms to the left. Keep the diagonal coefficients positive. Store the unknowns in the order `[u, v]`.

Complete A, b, and `p = np.linalg.solve(...)`. Use the solver directly; forming an inverse is unnecessary.


**Predict before running.** Write the two averaging equations. Which interior node should have the larger probability, and why?

*Write your prediction here.*


In [ ]:
A = ...
b = ...
p = ...


In [ ]:
ready(A,b,p)
assert np.array_equal(A,np.array([[2.,-1.],[-1.,2.]])), "Multiply each averaging equation by 2; keep unknowns ordered u, v."
assert np.array_equal(b,np.array([1.,0.])), "Move known endpoint contributions to the right."
assert np.allclose(A @ p,b)
assert np.isclose(p[0],(1+p[1])/2) and np.isclose(p[1],p[0]/2)
plt.plot(range(4),np.r_[1,p,0],"o-"); plt.xticks(range(4),["red","u","v","blue"])
plt.ylabel("Probability of reaching red first"); plt.ylim(-.05,1.05); plt.show()
print("Probabilities:",p,"Check passed.")


**Explain.** Explain each row of A and each entry of b in terms of the path. What would change in b if you asked for the probability of reaching blue first?

*Write your response here.*


**One elimination step.** The code below subtracts a multiple of the first equation from the second to eliminate u. Complete the two back-substitution expressions: solve the last equation for v, then the first for u. Use entries of U and c rather than typing numerical answers.


In [ ]:
U=A.copy(); c=b.copy()
factor=U[1,0]/U[0,0]
U[1]-=factor*U[0]
c[1]-=factor*c[0]
print("Triangular matrix:",U,"Right-hand side:",c)
v = ...
u = ...


In [ ]:
ready(u,v)
assert np.allclose([u,v],p), "Solve the last equation first, then use its value in the first."
assert np.allclose(U @ p,c), "The row operation preserves the solution."
print("Check passed.")


**Explain.** Why must the right-hand side be updated along with the coefficient row? Which feature of U makes back substitution possible?

*Write your response here.*


## 2. Decide whether a solution is unique

For a square full-rank matrix, the system has exactly one solution. A rank-deficient matrix can give infinitely many solutions or none, depending on b.

Inspect the three systems below. Use `np.linalg.matrix_rank` on A and on the augmented matrix `np.column_stack([A,b])`. A larger augmented rank indicates inconsistency. Classify each case using `"unique"`, `"many"`, or `"none"`.


In [ ]:
cases = [
    (np.array([[1.,1.],[1.,-1.]]), np.array([2.,0.])),
    (np.array([[1.,1.],[2.,2.]]), np.array([2.,4.])),
    (np.array([[1.,1.],[2.,2.]]), np.array([2.,5.])),
]
for i,(matrix,rhs) in enumerate(cases):
    print(i, "A =", matrix, "b =",rhs,sep="\n")


**Predict before running.** In the last two cases, what is the relationship between the coefficient rows? What geometric difference does changing the second right-hand side make?

*Write your prediction here.*


In [ ]:
labels = ...  # List the three classifications in case order.
for matrix,rhs in cases:
    print("rank(A):", np.linalg.matrix_rank(matrix),
          "rank([A|b]):", np.linalg.matrix_rank(np.column_stack([matrix,rhs])))


In [ ]:
ready(labels)
assert labels == ["unique","many","none"], "Compare the coefficient and augmented ranks."
for i,(matrix,rhs) in enumerate(cases):
    try:
        print("Case",i,"solver result:",np.linalg.solve(matrix,rhs))
    except np.linalg.LinAlgError as error:
        print("Case",i,"solver message:",error)
print("Check passed.")


**Explain.** Describe the two lines in each case. Why does a singular-matrix error alone fail to distinguish no solutions from infinitely many?

*Write your response here.*


## 3. Measure residual and solution error separately

For a computed or proposed solution x̂, the residual is `b - A @ x_hat`. It measures how closely x̂ satisfies the equations. The solution error compares x̂ with a known true solution.

Write `quality(A,b,x_hat,x_true)` to return `(relative_residual, relative_error)`, using Euclidean lengths (`np.linalg.norm`). Divide the residual length by the length of b, and the error length by the length of `x_true`. Both denominators are nonzero here.


**Predict before running.** Could a point far from the true intersection still nearly satisfy two nearly parallel lines? Explain before computing.

*Write your prediction here.*


In [ ]:
def quality(A,b,x_hat,x_true):
    relative_residual = ...
    relative_error = ...
    return relative_residual, relative_error


In [ ]:
q = quality(np.eye(2),np.array([3.,4.]),np.zeros(2),np.array([3.,4.]))
ready(*q)
assert np.allclose(q,(1,1)), "Check the denominators in both relative quantities."
q = quality(np.diag([2.,1.]),np.array([2.,1.]),np.array([2.,1.]),np.ones(2))
assert np.allclose(q,(2/np.sqrt(5),1/np.sqrt(2))), "Residual and error generally have different values."
records=[]
for eps in [1.,.01,1e-6]:
    matrix=np.array([[1.,1.],[1.,1.+eps]])
    true=np.ones(2); rhs=matrix @ true
    candidate=np.array([2.,0.])
    residual,error=quality(matrix,rhs,candidate,true)
    changed=np.linalg.solve(matrix,rhs+np.array([0.,1e-4]))
    records.append({"eps":eps,"condition number":np.linalg.cond(matrix),
                    "candidate relative residual":residual,"candidate relative error":error,
                    "solution change after perturbation":np.linalg.norm(changed-true)})
display(pd.DataFrame(records))
print("Check passed.")


**Explain.** Use the table to explain how a small residual can coexist with a large error. What does a large condition number warn about? Does it mean every computed solution must have a large error?

*Write your response here.*


## 4. Extend the path using sparse storage

With N interior nodes, each equation has coefficient 2 on its own unknown and -1 on each neighbouring interior unknown. The red endpoint contributes 1 to the first right-hand side; the blue endpoint contributes 0 to the last.

Build this tridiagonal matrix directly using `diags([lower, main, upper], [-1,0,1], shape=(N,N), format="csr")`. Use lengths N-1, N, and N-1 for the diagonals. Return the matrix and right-hand side. Assume N is at least 2.

CSR stores values in `data`, their column indices in `indices`, and row boundaries in `indptr`. COO stores row/column/value triples; CSC groups entries by column. Sparse direct solvers often prefer CSC.


**Predict before running.** For 1,000 interior nodes, about how many entries can be nonzero in a typical row? How does the total number of stored entries grow with N?

*Write your prediction here.*


In [ ]:
def path_system(N):
    matrix = ...
    rhs = ...
    return matrix, rhs


In [ ]:
for N in [2,5,17]:
    matrix,rhs=path_system(N)
    ready(matrix,rhs)
    assert matrix.format == "csr" and matrix.shape == (N,N)
    expected=2*np.eye(N)-np.eye(N,k=1)-np.eye(N,k=-1)
    assert np.allclose(matrix.toarray(),expected), "Use diagonal 2 and neighbouring coefficients -1."
    assert np.array_equal(rhs,np.r_[1.,np.zeros(N-1)]), "Only the first equation touches the red endpoint."
    assert matrix.nnz == 3*N-2
small_A,small_b=path_system(5)
print("Dense view of a small example:",small_A.toarray(),sep="\n")
print("data:",small_A.data,"\nindices:",small_A.indices,"\nindptr:",small_A.indptr)
print("Check passed.")


**Explain.** Use `indptr` to locate row 2 in `data` and `indices`. Explain how those entries encode that node’s equation. Why do the first and last rows have fewer nonzeros?

*Write your response here.*


## 5. Compute a product from CSR arrays

Write `csr_product(A,v)` with one loop over rows. For row i, use `A.indptr[i:i+2]` to locate its stored values and column indices, select the matching entries of v, and take their dot product. Do not convert A to dense or call `A @ v` inside your function.

The product takes O(nnz + N) work: one multiply-add per stored entry, plus work for N rows. This includes empty rows.


**Predict before running.** What should your function return for a row with no stored entries? Which elements of v are needed for one interior row of the path matrix?

*Write your prediction here.*


In [ ]:
def csr_product(A,v):
    out = np.zeros(A.shape[0])
    for i in range(A.shape[0]):
        start, stop = A.indptr[i:i+2]
        out[i] = ...
    return out


In [ ]:
tests=[(small_A,np.arange(5.)),
       (csr_matrix([[0.,2.,0.,-1.],[0.,0.,0.,0.],[3.,0.,4.,0.]]),np.array([2.,3.,5.,7.]))]
for matrix,vector in tests:
    answer=csr_product(matrix,vector)
    assert answer.shape == (matrix.shape[0],)
    assert np.allclose(answer,matrix @ vector), "Match each stored value to v at its stored column index."
print("Check passed, including an empty row and a rectangular matrix.")


**Explain.** Why does multiplying a dense matrix visit zero entries too? Explain why sparse indexing has a cost and why sparse storage may lose its advantage when most entries are nonzero.

*Write your response here.*


## 6. Solve the larger path and compare storage

Use `spsolve` on the matrix below. Compute actual CSR storage from the three arrays' `.nbytes`, and estimate dense storage from its shape and floating-point item size. Do not allocate the dense version.

Check the resulting probabilities against the original averaging equations. This uniform path also has a simple linear probability profile, which provides an independent check.


In [ ]:
N = 2000
large_A,large_b=path_system(N)


**Predict before running.** Sketch how the probabilities should change from the red end to the blue end. Predict whether the dense-to-sparse storage ratio grows or shrinks as N increases.

*Write your prediction here.*


In [ ]:
p_sparse = ...
csr_bytes = ...
dense_bytes = ...


In [ ]:
ready(p_sparse,csr_bytes,dense_bytes)
assert np.allclose(large_A @ p_sparse,large_b)
with_ends=np.r_[1.,p_sparse,0.]
assert np.allclose(p_sparse,(with_ends[:-2]+with_ends[2:])/2), "Check each node's averaging equation."
assert np.allclose(p_sparse,1-np.arange(1,N+1)/(N+1)), "The uniform path should have a linear profile."
assert csr_bytes == sum(a.nbytes for a in [large_A.data,large_A.indices,large_A.indptr])
assert dense_bytes == N*N*large_A.dtype.itemsize
print("Stored nonzeros:",large_A.nnz,"CSR bytes:",csr_bytes,"Dense bytes:",dense_bytes)
plt.plot(np.arange(N+2),with_ends); plt.xlabel("Node along path"); plt.ylabel("Probability of reaching red first"); plt.show()
print("Check passed.")


**Explain.** Compare the two storage amounts. Explain why matrix storage and matrix-vector products grow linearly for this path, while dense storage grows quadratically.

*Write your response here.*


## 7. Inspect what elimination can add

Sparse elimination can create new nonzero entries, called **fill-in**. The code below eliminates one variable from a small star-shaped system and compares the remaining nonzero patterns. Run it with the centre first and with a leaf first.

Elimination updates the remaining coefficients using the eliminated variable's row and column. The supplied formula performs that update; focus on which zero entries become nonzero.


**Predict before running.** The centre is connected to every leaf. A leaf is connected only to the centre. Which first elimination is more likely to connect previously unconnected leaves?

*Write your prediction here.*


In [ ]:
star=np.eye(6)
for leaf in range(1,6):
    star[0,0]+=1; star[leaf,leaf]+=1
    star[0,leaf]=star[leaf,0]=-1
fig,axes=plt.subplots(2,2,figsize=(7,6))
for row,pivot in enumerate([0,1]):
    keep=np.array([i for i in range(6) if i!=pivot])
    before=star[np.ix_(keep,keep)]
    after=before-np.outer(star[keep,pivot],star[pivot,keep])/star[pivot,pivot]
    new_entries=(np.abs(before)<1e-12)&(np.abs(after)>1e-12)
    print("Eliminate", "centre" if pivot==0 else "leaf", ": new nonzero entries =",new_entries.sum())
    for col,(label,matrix) in enumerate([("Before",before),("After",after)]):
        axes[row,col].imshow(np.abs(matrix)>1e-12,cmap="Blues",vmin=0,vmax=1)
        axes[row,col].set_title(("Centre" if pivot==0 else "Leaf")+" first: "+label)
plt.tight_layout(); plt.show()


**Explain.** Explain the difference between the two orders using the pictures. Why does the O(nnz + N) cost of one sparse matrix-vector product not determine the cost of a sparse direct solve? An iterative method performs repeated updates: what extra information would you need to estimate its total time?

*Write your response here.*


## Final check

Restart the runtime and run your completed notebook from the top. Check that every exercise passes, your figures appear, and every prediction and explanation is filled in. Save the notebook with its outputs.

**Source activities:** Meeting 13 linear systems and sparse-solvers activities, using the random-walk sequence of the Meeting 15 essentials deck. This practice version reorganizes those activities in the order of the Meeting 15 essentials deck.
